# Synthetic Research Engine — Calibration

Get **real validation numbers** for the synthetic research engine by
backtesting it against four datasets whose answers are already known:

| Backtest | Dataset | Measures |
|---|---|---|
| Choice-conjoint | `choiceData.csv` (72 real human choices) | Recovers real apple-type / freshness / price preference? |
| Ranking-conjoint | `conjointpizzadata.csv` | Rank correlation vs a known 1-16 ranking |
| Optimism | `AB_Test_Results.csv` | Does the panel wrongly favour the "new" variant? |
| Ignorance | `abtest1.csv` | Given undescribed options, does it stay near-uniform? |

Run the cells top to bottom. **Do not skip the connection test (step 5)** —
it catches a bad model name or key before the full ~3-minute run.

### 1. Install dependencies

In [ ]:
!pip install -q anthropic openai google-generativeai pandas numpy scipy

### 2. Get the project code

In [ ]:
import os
if os.path.isdir('/content/dashboard'):
    !cd /content/dashboard && git pull -q
else:
    !git clone -q https://github.com/joongsukkie/dashboard.git /content/dashboard
%cd /content/dashboard
!ls AB_Test_Results.csv abtest1.csv conjointpizzadata.csv choiceData.csv

### 3. Your AI key

**Set `PROVIDER` to the provider your key is from** — this is the #1 thing
people get wrong. The cell checks the key's format and warns if `PROVIDER`
and the key don't match.

In [ ]:
import getpass

PROVIDER = "openai"   # <-- SET THIS:  "openai" | "anthropic" | "gemini"

API_KEY = getpass.getpass(f"Paste your {PROVIDER} API key (hidden): ").strip()

def _guess_provider(k):
    if k.startswith("sk-ant-"): return "anthropic"
    if k.startswith("AIza"):    return "gemini"
    if k.startswith("sk-"):     return "openai"
    return "unknown"

_guess = _guess_provider(API_KEY)
if _guess != PROVIDER:
    print(f"\n*** MISMATCH ***")
    print(f"PROVIDER is set to '{PROVIDER}', but this key looks like a "
          f"'{_guess}' key.")
    print(f"Fix: change PROVIDER above to '{_guess}' (or paste a real "
          f"{PROVIDER} key), then re-run this cell.")
else:
    print(f"Key captured for {PROVIDER} ({len(API_KEY)} chars) — format looks right.")

### 4. Define the LLM caller

`MODELS` holds the model id per provider. If the connection test in step 5
fails with a "model not found" error, change the id here to one from the
list it prints, then re-run this cell and step 5.

In [ ]:
MODELS = {
    "openai":    "gpt-4o",
    "anthropic": "claude-sonnet-4-20250514",
    "gemini":    "gemini-1.5-pro",
}

def call_openai(api_key, prompt, strict=False):
    from openai import OpenAI
    client = OpenAI(api_key=api_key)
    system = "You are a data analytics expert. Return only valid JSON."
    if strict:
        system += " Return ONLY a JSON object, no other text, no markdown, no fences."
    resp = client.chat.completions.create(
        model=MODELS["openai"],
        messages=[{"role": "system", "content": system},
                  {"role": "user", "content": prompt}],
        temperature=0.2,
        response_format={"type": "json_object"},
    )
    return resp.choices[0].message.content

def call_anthropic(api_key, prompt, strict=False):
    import anthropic
    client = anthropic.Anthropic(api_key=api_key)
    system = "You are a data analytics expert. Return only valid JSON, no markdown fences, no prose."
    if strict:
        system += " CRITICAL: Return ONLY a JSON object starting with { and ending with }."
    resp = client.messages.create(
        model=MODELS["anthropic"], max_tokens=4096, system=system,
        messages=[{"role": "user", "content": prompt}],
    )
    return resp.content[0].text

def call_gemini(api_key, prompt, strict=False):
    import google.generativeai as genai
    genai.configure(api_key=api_key)
    system = "You are a data analytics expert. Return only valid JSON with no markdown fences."
    if strict:
        system += " Return ONLY a JSON object. No prose. No code fences."
    model = genai.GenerativeModel(
        MODELS["gemini"], system_instruction=system,
        generation_config={"response_mime_type": "application/json", "temperature": 0.2},
    )
    return model.generate_content(prompt).text

CALLERS = {"openai": call_openai, "anthropic": call_anthropic, "gemini": call_gemini}
caller = CALLERS[PROVIDER]
print(f"Caller ready: {PROVIDER} -> {MODELS[PROVIDER]}")

### 5. Connection test — RUN THIS BEFORE THE FULL CALIBRATION

It lists the models your key can use, then makes one real call with the
configured model. If it prints **SUCCESS**, go to step 6. If it fails,
copy a valid model id from the printed list into `MODELS` in step 4,
re-run step 4, then re-run this cell.

In [ ]:
print(f"=== Models available to your {PROVIDER} key ===")
try:
    if PROVIDER == "openai":
        from openai import OpenAI
        ids = sorted(m.id for m in OpenAI(api_key=API_KEY).models.list().data)
        for i in ids:
            if any(t in i for t in ("gpt", "o1", "o3", "o4")):
                print("  ", i)
    elif PROVIDER == "anthropic":
        import anthropic
        for m in anthropic.Anthropic(api_key=API_KEY).models.list().data:
            print("  ", m.id)
    elif PROVIDER == "gemini":
        import google.generativeai as genai
        genai.configure(api_key=API_KEY)
        for m in genai.list_models():
            if "generateContent" in m.supported_generation_methods:
                print("  ", m.name)
except Exception as e:
    print("  (could not list models:", e, ")")

print(f"\n=== Test call with MODELS['{PROVIDER}'] = {MODELS[PROVIDER]} ===")
try:
    out = caller(API_KEY, 'Return this exact JSON and nothing else: {"ok": true}')
    print("SUCCESS — the model responded:")
    print(" ", out[:300])
except Exception as e:
    import traceback
    traceback.print_exc()
    print("\n>>> FAILED. Read the error above:")
    print("    - 'invalid x-api-key' / 401  -> PROVIDER (step 3) does not match your key")
    print("    - 'model not found' / 404    -> pick a valid id from the list, set it in MODELS (step 4)")

### 6. Run the calibration

All backtests, ~14 LLM calls, ~3-4 minutes. The choice-conjoint test
runs twice — **ungrounded** vs **RAG-grounded** — so you see how much
grounding the engine in real records fixes the name-bias failure.

In [ ]:
# Reload modules so re-runs in this kernel use the latest pulled code.
import importlib, rag, evidence, synthetic_research, calibration
for _m in (rag, evidence, synthetic_research, calibration):
    importlib.reload(_m)

profile = calibration.run_calibration(caller, API_KEY)   # writes calibration_profile.json
calibration._print_report(profile)

### 7. Per-backtest detail

In [ ]:
print("TRUST GRADES      :", profile.get("trust"))
print("RECOMMENDED FIXES :", profile.get("recommended_corrections"))
print()
for r in profile.get("results", []):
    print("-" * 72)
    print(r["case"])
    print("  context:", r["note"])
    print("  scores :", {k: v for k, v in r["score"].items()
                          if k not in ("ok", "predicted_order")})

### 8. Download the calibration profile

`calibration_profile.json` is what the live app reads to attach an honest
trust level to every synthetic result.

In [ ]:
from google.colab import files
files.download("calibration_profile.json")

### How to read the numbers

**Choice-conjoint** (the strongest test — real discrete-choice experiment)
- `type_spearman` / `freshness_spearman` — rank correlation with real human
  preference. `1.0` = perfect order, `0` = none, negative = backwards.
- `price_sign_correct` — did the engine prefer the cheaper option? (want `True`)
- `top_type_correct` / `top_freshness_correct` — picked the real favourite?

**Comparison**
- `optimism_bias` — how much the panel over-favours the "new" option (the
  AB test's control actually won, so positive = optimism bias).
- `ignorance_overconfidence` — distance from a uniform split on undescribed
  options. Near `0` = well-calibrated; large = it invents signal.

Trust grades flow into the live app: a study's confidence is capped at the
grade its backtest family earned — the engine can't claim more certainty
than it proved.

If a backtest still shows `"Every segment LLM call failed"`, the error
string now names the cause — model-not-found, bad key, or rate limit.